# Japon (TOPIX 500) et Hong Kong : jusqu'où vont les actions ?

Même test que `japan_stocks_check.ipynb`, pour 3 univers :

| run | groupe | bourse | actions |
|---|---|---|---|
| Japon, Nikkei 225 | 1 | Tokyo | `japan_returns_nikkei225.csv` (le test précédent) |
| Japon, TOPIX 500 | 1 | Tokyo | `japan_returns.csv` |
| Hong Kong | 2 | Hong Kong | `hongkong_returns.csv` |

Pour chaque run : appris avant 2013, mesuré sur 2013-2015, dates connues (`calendar/train_dates.csv`).

Avant : `python market_data/download_stocks.py japan` puis `python market_data/download_stocks.py hongkong`.

⚠️ Pas une prévision : on utilise les vrais rendements du jour de la target.

In [1]:
import numpy as np
import pandas as pd
import exchange_calendars as xcals
import warnings
warnings.filterwarnings("ignore")

X = pd.read_csv("data/X_train.csv", index_col="ROW_ID")
y = pd.read_csv("data/y_train.csv", index_col="ROW_ID")["target"]
dates = pd.read_csv("calendar/train_dates.csv", parse_dates=["date"])
index_returns = pd.read_csv("market_data/returns.csv", index_col="Date", parse_dates=True)

dates = dates[(dates["method"] == "holidays") & dates["date"].notna()]
X["date"] = X["TS"].map(dict(zip(dates["TS"], dates["date"])))
X["target"] = y
X = X.dropna(subset=["date"])
hist_cols = [f"RET_{i}" for i in range(20, 0, -1)]   # oldest -> newest

## Les fonctions

In [2]:
def prepare(group, exchange, index_name, returns_file):
    # rows of the group, next session (= day of the target), stock returns on the exchange calendar
    Xg = X[X["GROUP"] == group].copy()
    sessions = pd.DatetimeIndex(xcals.get_calendar(exchange, start="2004-01-01").sessions)
    Xg["next_date"] = sessions[sessions.searchsorted(Xg["date"], side="right")]

    stocks = pd.read_csv(returns_file, index_col="Date", parse_dates=True)
    period = (stocks.index >= Xg["date"].min()) & (stocks.index <= Xg["next_date"].max())
    coverage = stocks.loc[period].notna().mean()
    keep = list(coverage[coverage >= 0.9].index)

    # cap a few bad prints, put everything on the exchange calendar
    R = stocks[keep].clip(-0.3, 0.3).reindex(sessions).fillna(0)
    R[index_name] = index_returns[index_name].reindex(sessions).fillna(0)
    fit_days = sessions[(sessions >= Xg["date"].min()) & (sessions < "2013-01-01")]
    R = R / R.loc[fit_days].std().replace(0, 1)

    print(f"group {group}: {len(Xg)} rows, {Xg['ALLOCATION'].nunique()} allocations, "
          f"{Xg['date'].min().date()} -> {Xg['date'].max().date()} | "
          f"{stocks.shape[1]} stocks downloaded, {len(keep)} cover the period")
    return Xg, sessions, R, keep


def ridge(A, b, lam):
    return np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ b)


def fit_and_predict(Xg, R, columns, lam, fit_start, fit_end, pred_start, pred_end):
    pred = pd.Series(np.nan, index=Xg.index)
    betas = {}
    for alloc, sub in Xg.groupby("ALLOCATION"):
        rows_fit = sub[(sub["date"] >= fit_start) & (sub["date"] < fit_end)]
        rows_pred = sub[(sub["date"] >= pred_start) & (sub["date"] < pred_end)]
        if len(rows_fit) < 100:
            continue
        beta = ridge(R.loc[rows_fit["next_date"], columns].values, rows_fit["target"].values, lam)
        betas[alloc] = beta
        if len(rows_pred) > 0:
            pred[rows_pred.index] = R.loc[rows_pred["next_date"], columns].values @ beta
    return pred, betas


def accuracy(Xg, pred, mask):
    ok = mask & pred.notna()
    return ((pred[ok] > 0) == (Xg.loc[ok, "target"] > 0)).mean()


def r2(Xg, pred, mask):
    ok = mask & pred.notna()
    t = Xg.loc[ok, "target"]
    return 1 - ((t - pred[ok]) ** 2).sum() / ((t - t.mean()) ** 2).sum()


def date_scores(rows, pred_by_alloc, sessions):
    # score of every session as the date of these rows (drop of squared error on the 20-day history)
    total = np.zeros(len(sessions) - 19)
    for alloc, hist in zip(rows["ALLOCATION"], rows[hist_cols].values):
        if alloc not in pred_by_alloc:
            continue
        p = pred_by_alloc[alloc]
        h = np.nan_to_num(hist.astype(float))
        total += 2 * np.convolve(p, h[::-1], mode="valid") - np.convolve(p ** 2, np.ones(20), mode="valid")
    return pd.Series(total, index=sessions[19:])

In [3]:
def evaluate(name, group, exchange, index_name, returns_file, rolling=True):
    Xg, sessions, R, stock_cols = prepare(group, exchange, index_name, returns_file)
    start = Xg["date"].min()
    hold = Xg["date"] >= "2013-01-01"
    out = {"run": name, "stocks used": len(stock_cols)}

    # penalty chosen on 2012
    val = (Xg["date"] >= "2012-01-01") & (Xg["date"] < "2013-01-01")
    best_lam, best_r2 = None, -np.inf
    for lam in [100, 1000, 10000, 100000]:
        p, _ = fit_and_predict(Xg, R, stock_cols, lam, start, "2012-01-01", "2012-01-01", "2013-01-01")
        if r2(Xg, p, val) > best_r2:
            best_lam, best_r2 = lam, r2(Xg, p, val)
    out["lambda"] = best_lam

    # index alone vs stocks, fixed exposures learned before 2013
    p_index, betas_index = fit_and_predict(Xg, R, [index_name], 10, start, "2013-01-01", "2013-01-01", "2100-01-01")
    p_stocks, betas = fit_and_predict(Xg, R, stock_cols, best_lam, start, "2013-01-01", "2013-01-01", "2100-01-01")
    out["acc index"] = accuracy(Xg, p_index, hold)
    out["acc stocks fixed"] = accuracy(Xg, p_stocks, hold)
    out["R2 index"] = r2(Xg, p_index, hold)
    out["R2 stocks fixed"] = r2(Xg, p_stocks, hold)

    # exposures re-fitted every 20 days on the previous year (upper bound)
    if rolling:
        p_roll = pd.Series(np.nan, index=Xg.index)
        hold_days = sorted(Xg.loc[hold, "date"].unique())
        for k in range(0, len(hold_days), 20):
            a = hold_days[k]
            b = hold_days[k + 20] if k + 20 < len(hold_days) else pd.Timestamp("2100-01-01")
            p, _ = fit_and_predict(Xg, R, stock_cols, best_lam, a - pd.Timedelta(days=365), a, a, b)
            p_roll[p.notna()] = p[p.notna()]
        out["acc stocks rolling"] = accuracy(Xg, p_roll, hold)
        out["R2 stocks rolling"] = r2(Xg, p_roll, hold)
        strength = p_roll[hold].abs()
        out["acc rolling top 20%"] = accuracy(Xg, p_roll, hold & (p_roll.abs() >= strength.quantile(0.8)))

    # dating isolated rows of this group only (exposures learned before 2013)
    pred_index = {a: R[[index_name]].values @ b for a, b in betas_index.items()}
    pred_stocks = {a: R[stock_cols].values @ b for a, b in betas.items()}
    test_like = Xg.loc[hold].drop_duplicates("TS").sample(150, random_state=0)
    hits_index, hits_stocks = [], []
    for ts, true_date in zip(test_like["TS"], test_like["date"]):
        rows = Xg[Xg["TS"] == ts]
        hits_index.append(date_scores(rows, pred_index, sessions).idxmax() == true_date)
        hits_stocks.append(date_scores(rows, pred_stocks, sessions).idxmax() == true_date)
    out["dating index"] = np.mean(hits_index)
    out["dating stocks"] = np.mean(hits_stocks)
    return out

## Les 3 runs

In [4]:
import os

todo = [
    ("Japan, Nikkei 225", 1, "XTKS", "NIKKEI225", "market_data/japan_returns_nikkei225.csv"),
    ("Japan, TOPIX 500", 1, "XTKS", "NIKKEI225", "market_data/japan_returns.csv"),
    ("Hong Kong", 2, "XHKG", "HSCEI", "market_data/hongkong_returns.csv"),
]
runs = []
for name, group, exchange, index_name, returns_file in todo:
    if not os.path.exists(returns_file):
        print("skipped", name, ": run the download first ->", returns_file)
        continue
    runs.append(evaluate(name, group, exchange, index_name, returns_file))

group 1: 99890 rows, 70 allocations, 2007-09-26 -> 2015-10-30 | 238 stocks downloaded, 219 cover the period


group 1: 99890 rows, 70 allocations, 2007-09-26 -> 2015-10-30 | 499 stocks downloaded, 440 cover the period


group 2: 134858 rows, 70 allocations, 2008-01-04 -> 2015-10-30 | 166 stocks downloaded, 116 cover the period


In [5]:
summary = pd.DataFrame(runs).set_index("run").T
summary

run,"Japan, Nikkei 225","Japan, TOPIX 500",Hong Kong
stocks used,219.000000,440.000000,116.000000
lambda,1000.000000,1000.000000,1000.000000
acc index,0.557453,0.557453,0.566859
acc stocks fixed,0.590014,0.603577,0.593969
R2 index,0.084483,0.084483,0.083541
R2 stocks fixed,0.110823,0.127165,0.121392
acc stocks rolling,0.600930,0.614906,0.619308
R2 stocks rolling,0.161974,0.197293,0.171312
acc rolling top 20%,0.713149,0.743333,0.731988
dating index,0.020000,0.020000,0.173333
